# Day 40: Multi‑Agent Debate System

Two agents debate a question, then a judge produces a refined answer.

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Define agent personas

In [ ]:
AGENT_A_SYSTEM = """You are an optimistic, creative thinker. You believe in bold ideas and see possibilities everywhere.
When given a question, you will argue for the most innovative and hopeful answer.
Be enthusiastic and slightly unconventional."""

AGENT_B_SYSTEM = """You are a critical, analytical thinker. You focus on logical rigor, evidence, and potential flaws.
When given a question, you will challenge assumptions, point out weaknesses, and argue for a cautious, well‑reasoned answer.
Be precise and skeptical."""

JUDGE_SYSTEM = """You are a neutral judge. You have observed a debate between two agents.
Your task is to synthesise the best of both arguments into a single, balanced, accurate answer.
You should be fair, cite the strong points from each side, and produce a final refined answer."""

## 2. Debate loop

In [ ]:
def call_llm(messages, system_prompt=None):
    if system_prompt:
        messages = [{"role": "system", "content": system_prompt}] + messages
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content

def debate(question, rounds=3):
    history_a = []
    history_b = []
    
    for round_num in range(rounds):
        # Agent A speaks
        if round_num == 0:
            prompt = f"Question: {question}\nProvide your initial argument."
        else:
            prompt = f"Question: {question}\n\nPrevious argument from Agent B: {history_b[-1]}\n\nProvide your rebuttal and refine your position."
        a_response = call_llm([{"role": "user", "content": prompt}], AGENT_A_SYSTEM)
        history_a.append(a_response)
        
        # Agent B speaks
        prompt = f"Question: {question}\n\nAgent A said: {a_response}\n\nProvide your critical response."
        b_response = call_llm([{"role": "user", "content": prompt}], AGENT_B_SYSTEM)
        history_b.append(b_response)
        
        print(f"\n=== Round {round_num+1} ===")
        print(f"Agent A (optimistic): {a_response[:200]}...")
        print(f"Agent B (critical): {b_response[:200]}...")
    
    return history_a, history_b

## 3. Judge synthesis

In [ ]:
def judge_debate(question, history_a, history_b):
    # Create a transcript
    transcript = f"Question: {question}\n\n"
    for i, (a, b) in enumerate(zip(history_a, history_b)):
        transcript += f"Round {i+1}:\nAgent A: {a}\nAgent B: {b}\n\n"
    
    judge_prompt = f"""Here is the debate transcript:
{transcript}

Based on the debate, produce a final refined answer to the original question.
Synthesise the strengths of both sides and present a balanced, accurate conclusion."""
    
    final_answer = call_llm([{"role": "user", "content": judge_prompt}], JUDGE_SYSTEM)
    return final_answer

def run_debate_system(question, rounds=3):
    print(f"Debating: {question}\n")
    history_a, history_b = debate(question, rounds)
    final = judge_debate(question, history_a, history_b)
    print("\n=== FINAL JUDGEMENT ===")
    print(final)
    return final

## 4. Example

In [ ]:
question = "Should we invest heavily in artificial general intelligence (AGI) research?"
result = run_debate_system(question, rounds=2)  # fewer rounds for demo